In [3]:
import pandas as pd
from xgboost import XGBRegressor
from sklearn.metrics import r2_score

# Load saved training and test CSV files from week 3
train = pd.read_csv("data/cleaned_train.csv")
test = pd.read_csv("data/cleaned_test.csv")

print("Training shape: ", train.shape)
print("Test shape: ", test.shape)

Training shape:  (117914, 828)
Test shape:  (12789, 828)


Based on results from previous weeks, I narrowed down to focus on the last two feature sets for this week's Gradient Boosting model evaluation.

In [2]:
target = "ClosePrice"

city_cols = [
    col for col in train.columns
    if col.startswith("City_grouped_")
]

postal_cols = [
    col for col in train.columns
    if col.startswith("PostalCode_grouped_")
]

county_cols = [
    col for col in train.columns 
    if col.startswith("CountyOrParish_")
]

school_district_cols = [
    col for col in train.columns 
    if col.startswith("SchoolDistrict_")
]

features_sets = {
    "With Engineered Features + All Location + SchoolDistrict": [
        "LivingArea",
        "BedroomsTotal",
        "BathroomsTotalInteger",
        "LotSizeSquareFeet",
        "Missing_LotSizeSquareFeet",
        "ViewYN",
        "WaterfrontYN",
        "BasementYN",
        "PoolPrivateYN",
        "PropertyAge",
        "BedBathRatio"
    ]    + city_cols + postal_cols + county_cols + school_district_cols,

    "With Everything + Missing_YearBuilt": [
        "LivingArea",
        "BedroomsTotal",
        "BathroomsTotalInteger",
        "LotSizeSquareFeet",
        "Missing_LotSizeSquareFeet",
        "ViewYN",
        "WaterfrontYN",
        "BasementYN",
        "PoolPrivateYN",
        "PropertyAge",
        "Missing_YearBuilt",
        "BedBathRatio"
    ]    + city_cols + postal_cols + county_cols + school_district_cols

}


XGBoost Model Basline
- Default values for depth, learning rate, and n_estimators (parameters for XGBRegressor).

In [9]:
xgboost_results = []

for name, features in features_sets.items():

    # Define features and target
    X_train = train[features]   # Training Set
    # print(name, X_train.shape)
    y_train = train[target]

    X_test = test[features]     # Test Set
    y_test = test[target]

    # Model
    model = XGBRegressor(random_state=28)      # Apply the XG Boost Model with default parameter values
    model.fit(X_train, y_train)     # Train model

    # Predictions on target variable
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    # Compute R^2 scores
    r2_train = r2_score(y_train, y_pred_train)
    r2_test = r2_score(y_test, y_pred_test)

    # Save results
    xgboost_results.append({
        "Feature Set": name,
        "Number of Features": len(features),
        "Training R^2": round(r2_train, 6),
        "Test R^2": round(r2_test, 6)
    })

xgboost_results_df = pd.DataFrame(xgboost_results)
xgboost_results_df

,Feature Set,Number of Features,Training R^2,Test R^2
0,With Engineered Features + All Location + Scho...,795,0.850954,0.814515
1,With Everything + Missing_YearBuilt,796,0.851390,0.814688


Hyperparameter Tuned Models
- n_estimators: total number of trees
- max_depth: maximum tree complexity (levels) of each tree
- learning_rate: step-size scaling factor for each prediction/ boosting iteration 
    - higher value -> faster learning

In [26]:
param_settings = {
    'n_estimators': [100, 250, 500],
    'max_depth': [3, 5, 8],
    'learning_rate': [0.02, 0.05, 0.1]
}

tuned_results = []

for name, features in features_sets.items():

    X_train = train[features]   # Training Set
    y_train = train[target]
    
    X_test = test[features]     # Test Set
    y_test = test[target]
    
    for n_estimator in param_settings['n_estimators']:
        for max_depth in param_settings['max_depth']:
            for learning_rate in param_settings['learning_rate']:
            
                model = XGBRegressor(
                    n_estimators=n_estimator,
                    max_depth=max_depth,
                    learning_rate=learning_rate,
                    random_state=28
                )

                model.fit(X_train, y_train)     # Train model

                # Prediction
                y_pred_train = model.predict(X_train)
                y_pred_test = model.predict(X_test)
            
                # Compute R^2 scores
                r2_train = r2_score(y_train, y_pred_train)
                r2_test = r2_score(y_test, y_pred_test)
            
                # Save results
                tuned_results.append({
                    "Feature Set": name,
                    "n_estimators": n_estimator,
                    "max_depth": max_depth,
                    "learning_rate": learning_rate,
                    "Training R^2": round(r2_train, 6),
                    "Test R^2": round(r2_test, 6)
                })

In [27]:
tuned_results_df = pd.DataFrame(tuned_results).sort_values(by='Test R^2', ascending=False).reset_index(drop=True)
tuned_results_df.head(6)

,Feature Set,n_estimators,max_depth,learning_rate,Training R^2,Test R^2
0,With Everything + Missing_YearBuilt,500,8,0.10,0.903713,0.835624
1,With Engineered Features + All Location + Scho...,500,8,0.10,0.904204,0.834611
2,With Everything + Missing_YearBuilt,500,8,0.05,0.876006,0.822118
3,With Engineered Features + All Location + Scho...,500,8,0.05,0.874771,0.820760
4,With Everything + Missing_YearBuilt,250,8,0.10,0.877731,0.820710
5,With Engineered Features + All Location + Scho...,250,8,0.10,0.878388,0.820459


In [28]:
tuned_results_df.tail(5)

,Feature Set,n_estimators,max_depth,learning_rate,Training R^2,Test R^2
49,With Engineered Features + All Location + Scho...,250,3,0.02,0.649614,0.638912
50,With Everything + Missing_YearBuilt,100,5,0.02,0.629335,0.615053
51,With Engineered Features + All Location + Scho...,100,5,0.02,0.629335,0.615053
52,With Engineered Features + All Location + Scho...,100,3,0.02,0.538090,0.530193
53,With Everything + Missing_YearBuilt,100,3,0.02,0.538090,0.530193


Results Summary
- Without specifying the hyperparameters, XGBoost model gives the highest R^2 comparing with all three models trained previously.

- After tuning, best R^2 was observed for the most comprehensive feature set used and with the parameters combination n_estimators=500, max_depth=8, learning_rate=0.1
    - Training R^2: 0.903713
    - Test R^2: 0.835624

In [34]:
summary_table = {
    'Feature Set': ['With Engineered Features + All Location', 'With Everything + Missing_YearBuilt'],
    'Linear Regression': [0.440762, 0.440762],
    'Decision Tree Regressor': [0.650386, 0.652362],
    'Random Forest Regressor': [0.810480, 0.810361],
    'XGBoost (Baseline)': [0.814515, 0.814688],
    'XGBoost (Tuned)': [0.834611, 0.835624]
}
pd.DataFrame(summary_table)

,Feature Set,Linear Regression,Decision Tree Regressor,Random Forest Regressor,XGBoost (Baseline),XGBoost (Tuned)
0,With Engineered Features + All Location,0.440762,0.650386,0.810480,0.814515,0.834611
1,With Everything + Missing_YearBuilt,0.440762,0.652362,0.810361,0.814688,0.835624


- Except for Linear Regression and Random Forest Regressor models, most comprehensive feature set (includes missing YearBuilt) has a slighlty higher test R^2 (<0.01 difference) than the the set excluding the flagged YearBuilt information.